# 01 - Data Collection
Mengambil data dari World Bank API menggunakan library `wbgapi`.

**Indikator yang diambil:**
- GDP Growth (NY.GDP.MKTP.KD.ZG)
- Inflasi (FP.CPI.TOTL.ZG)
- Pengangguran (SL.UEM.TOTL.ZS)
- Pertumbuhan Populasi (SP.POP.GROW)
- Ekspor % GDP (NE.EXP.GNFS.ZS)
- Impor % GDP (NE.IMP.GNFS.ZS)
- FDI % GDP (BX.KLT.DINV.WD.GD.ZS)
- Nilai Tukar (PA.NUS.FCRF)

In [ ]:
import wbgapi as wb
import pandas as pd
import os

# Indikator World Bank
INDICATORS = {
    'NY.GDP.MKTP.KD.ZG': 'GDP_Growth',
    'FP.CPI.TOTL.ZG':    'Inflation',
    'SL.UEM.TOTL.ZS':    'Unemployment',
    'SP.POP.GROW':       'Population_Growth',
    'NE.EXP.GNFS.ZS':    'Exports',
    'NE.IMP.GNFS.ZS':    'Imports',
    'BX.KLT.DINV.WD.GD.ZS': 'FDI',
    'PA.NUS.FCRF':       'Exchange_Rate'
}

COUNTRY = 'IDN'  # Indonesia
START_YEAR = 1991
END_YEAR = 2024

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Mengambil data dari World Bank API...')

: 

In [ ]:
# Ambil setiap indikator
dfs = []

for code, name in INDICATORS.items():
    print(f'Mengambil: {name} ({code})')
    try:
        df = wb.data.DataFrame(
            code,
            economy=COUNTRY,
            time=range(START_YEAR, END_YEAR + 1)
        )
        df = df.T.reset_index()
        df.columns = ['Year', name]
        df['Year'] = df['Year'].str.replace('YR', '').astype(int)
        df.to_csv(f'../data/raw/{name}.csv', index=False)
        dfs.append(df)
        print(f'  -> {len(df)} baris, missing: {df[name].isna().sum()}')
    except Exception as e:
        print(f'  -> ERROR: {e}')

print('\nSemua indikator berhasil diambil!')

In [ ]:
# Gabungkan semua indikator
from functools import reduce

dataset = reduce(lambda left, right: pd.merge(left, right, on='Year', how='outer'), dfs)
dataset = dataset.sort_values('Year').reset_index(drop=True)

# Simpan
dataset.to_csv('../data/processed/dataset_indonesia.csv', index=False)

print('Dataset gabungan:')
print(dataset.shape)
dataset.head(10)

In [ ]:
# Cek missing values
print('Missing values per kolom:')
missing = dataset.isnull().sum()
missing_pct = (missing / len(dataset) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})

In [ ]:
# Statistik deskriptif
dataset.describe().round(3)